# scene3d — All-in-One Colab Notebook
**Video → Frames → MASt3R-SLAM → gsplat → SAM2 → Semantic Lifting → CLIP → Download**

Everything runs here. The only step left for your Mac is launching the viewer.

### Before running
1. Runtime → Change runtime type → **T4 GPU** (free) or A100 (Pro)
2. Upload your video to Google Drive at:
   `MyDrive/scene3d/{SCENE_NAME}/input.mp4`
3. Set `SCENE_NAME` in the config cell below
4. Run All (Runtime → Run all)

### After running
Download from Drive: `MyDrive/scene3d/{SCENE_NAME}/outputs/`  
Place on Mac as: `outputs/{SCENE_NAME}/` and `data/{SCENE_NAME}/`  
Then: `python -m viewer.app --scene {SCENE_NAME}`

**Expected total time:** ~90 min on T4 | ~50 min on A100

---
## 0. Mount Drive & Configure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, glob, struct, json, pathlib, subprocess
import numpy as np
import cv2

# ── EDIT THIS ────────────────────────────────────────────────
SCENE_NAME = "scene1"
# ─────────────────────────────────────────────────────────────

# Input video on Drive
VIDEO_ON_DRIVE = f"/content/drive/MyDrive/scene3d/{SCENE_NAME}/input.mp4"

# All intermediate work on Colab local SSD (fast I/O)
WORK_DIR      = f"/content/scene3d_work/{SCENE_NAME}"
FRAMES_DIR    = f"{WORK_DIR}/frames"
SLAM_DIR      = "/content/MASt3R-SLAM"
SLAM_LOGS     = f"{SLAM_DIR}/logs"
COLMAP_DIR    = f"{WORK_DIR}/colmap"
GSPLAT_REPO   = "/content/gsplat"
GSPLAT_OUTPUT = f"{WORK_DIR}/gsplat_output"
MASKS_DIR     = f"{WORK_DIR}/masks"
SEMANTIC_PLY  = f"{WORK_DIR}/splat_semantic.ply"
EMBEDDINGS    = f"{WORK_DIR}/embeddings.npz"

# Final outputs saved back to Drive
DRIVE_OUTPUT  = f"/content/drive/MyDrive/scene3d/{SCENE_NAME}/outputs"

# Runtime globals
SLAM_TRAJ = None
SLAM_PLY  = None
FINAL_PLY = None

os.makedirs(WORK_DIR, exist_ok=True)
print(f"Scene : {SCENE_NAME}")
print(f"Video : {VIDEO_ON_DRIVE}")
print(f"Work  : {WORK_DIR}")

---
## 1. Extract & Filter Frames
Runs ffmpeg directly on Colab — no Mac preprocessing needed.

In [ ]:
!pip install -q scikit-image  # explicit install — not guaranteed on all Colab runtimes
from skimage.metrics import structural_similarity as ssim

os.makedirs(FRAMES_DIR, exist_ok=True)

VIDEO_LOCAL = f"{WORK_DIR}/input.mp4"
if not os.path.exists(VIDEO_LOCAL):
    print("Copying video from Drive...")
    shutil.copy2(VIDEO_ON_DRIVE, VIDEO_LOCAL)
print(f"Video: {os.path.getsize(VIDEO_LOCAL)/1e6:.1f} MB")

# Stage 1: Extract frames at 3 FPS, 518px height
print("\nExtracting frames...")
result = subprocess.run([
    "ffmpeg", "-i", VIDEO_LOCAL,
    "-vf", "fps=3,scale=-1:518",
    "-q:v", "2", "-y",
    os.path.join(FRAMES_DIR, "%06d.jpg")
], capture_output=True, text=True)

if result.returncode != 0:
    raise RuntimeError(f"ffmpeg failed:\n{result.stderr[-500:]}")

frames = sorted(glob.glob(os.path.join(FRAMES_DIR, "*.jpg")))
print(f"  Extracted {len(frames)} frames")
if len(frames) == 0:
    raise RuntimeError("ffmpeg ran but produced 0 frames. Check the video file path.")

# Stage 2: Blur detection
print("\nFiltering blurry frames...")
scores = []
for f in frames:
    gray = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    scores.append(cv2.Laplacian(gray, cv2.CV_64F).var())

threshold = np.percentile(scores, 5)
kept, removed = [], 0
for f, s in zip(frames, scores):
    if s >= threshold:
        kept.append(f)
    else:
        os.remove(f)
        removed += 1
frames = kept
print(f"  Removed {removed} blurry frames (threshold={threshold:.1f})")

if len(frames) == 0:
    raise RuntimeError("All frames were removed by the blur filter — check video quality.")

# Stage 3: Deduplication (SSIM)
print("\nRemoving duplicate frames...")
if len(frames) == 1:
    kept = frames
else:
    kept = [frames[0]]
    prev = cv2.imread(frames[0], cv2.IMREAD_GRAYSCALE)
    removed = 0
    for f in frames[1:-1]:
        curr = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if ssim(prev, curr) < 0.98:
            kept.append(f)
            prev = curr
        else:
            os.remove(f)
            removed += 1
    kept.append(frames[-1])
    frames = kept
    print(f"  Removed {removed} duplicate frames")

# Stage 4: Rename sequentially + convert to .png (MASt3R-SLAM requires .png)
print("\nRenaming + converting to .png...")
for i, old in enumerate(sorted(frames)):
    new_jpg = os.path.join(FRAMES_DIR, f"{i+1:06d}.jpg")
    new_png = os.path.join(FRAMES_DIR, f"{i+1:06d}.png")
    if old != new_jpg:
        os.rename(old, new_jpg)
    img = cv2.imread(new_jpg)
    cv2.imwrite(new_png, img)
    os.remove(new_jpg)

final_frames = sorted(glob.glob(os.path.join(FRAMES_DIR, "*.png")))
print(f"\n✓ {len(final_frames)} frames ready in {FRAMES_DIR}/")

---
## 2. Install VGGT (replaces MASt3R-SLAM)

**Why VGGT instead of MASt3R-SLAM?**  
MASt3R-SLAM requires compiling custom CUDA C++ extensions (`mast3r_slam_backends`)  
that consistently fail to build on Colab's environment.  
VGGT is **pure PyTorch** — no compilation needed — and is more accurate for short  
videos (CVPR 2025 Best Paper, processes all frames in one 30-second forward pass).

In [ ]:
import torch
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
print(f"Torch: {torch.__version__}")

In [ ]:
VGGT_DIR = "/content/vggt"

if not os.path.exists(VGGT_DIR):
    !git clone https://github.com/facebookresearch/vggt.git {VGGT_DIR}
    print("✓ VGGT cloned")
else:
    print("✓ VGGT already cloned")

!pip install -q -e {VGGT_DIR}
!pip install -q einops timm huggingface_hub plyfile

# VGGT's deps pull numpy down to 1.26.4 — force it back up
!pip install -q "numpy>=2.0"

print("✓ VGGT installed — no CUDA compilation required")

In [ ]:
import torch
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
print(f"Torch: {torch.__version__}")

In [ ]:
# VGGT-1B checkpoint is downloaded automatically on first use via HuggingFace Hub.
# ~1.1 GB, CC BY-NC-SA 4.0 license (non-commercial).
# Nothing to do here — the model loads in the next cell.
print("✓ VGGT checkpoint will be downloaded automatically from HuggingFace in the next cell")

---
## 3. Run VGGT — Geometry Estimation + COLMAP Export

VGGT processes all frames in **one forward pass** (~30 sec on T4) and directly  
writes the COLMAP workspace that gsplat needs.  
No SLAM loop, no chunking issues, no C++ backends.

In [ ]:
!pip install -q plyfile

import os, sys, torch, struct, shutil, glob
import numpy as np
import cv2
from PIL import Image as PILImage
import torchvision.transforms.functional as TF
from plyfile import PlyData, PlyElement

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
torch.backends.cuda.enable_math_sdp(True)   # reset in case a previous run disabled it
sys.path.insert(0, VGGT_DIR)

device = "cuda"
dtype  = torch.float32   # float32 throughout — avoids dtype mismatch with VGGT LayerNorm

# ── 1. Load VGGT-1B ─────────────────────────────────────────────────────────
print("Loading VGGT-1B (~1.1 GB on first run)...")
from vggt.models.vggt import VGGT
model = VGGT.from_pretrained("facebook/VGGT-1B")
model = model.to(device).eval()
model = model.float()   # ensure all weights are float32 regardless of session state
print("✓ VGGT-1B loaded")

# ── 2. Load frames at 224px ──────────────────────────────────────────────────
# 224 = 14 × 16 — valid ViT patch grid
# T4 (15 GB): model uses ~11 GB, leaving ~3 GB for activations
# At 224px: 17 frames × 256 patches = 4,352 tokens → ~2.3 GB activations → fits
IMAGE_SIZE = 224

image_paths = sorted(glob.glob(os.path.join(FRAMES_DIR, "*.png")))
if not image_paths:
    image_paths = sorted(glob.glob(os.path.join(FRAMES_DIR, "*.jpg")))
N = len(image_paths)
print(f"Frames: {N}  |  Resolution: {IMAGE_SIZE}px")

images_list = []
for path in image_paths:
    img = PILImage.open(path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    t = TF.to_tensor(img)
    t = TF.normalize(t, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    images_list.append(t)

images = torch.stack(images_list, dim=0).to(device, dtype=dtype)  # (N, 3, 224, 224)
print(f"Input shape: {tuple(images.shape)}")

# ── 3. VGGT inference ────────────────────────────────────────────────────────
print("Running VGGT inference...")
with torch.no_grad():
    predictions = model(images[None])  # (1, N, 3, H, W)
print("✓ VGGT inference complete")

# ── 4. Extract camera poses ──────────────────────────────────────────────────
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

# image_size_hw required: used to convert predicted FOV → focal length
cam_to_world, intrinsics_t = pose_encoding_to_extri_intri(
    predictions["pose_enc"],
    image_size_hw=(IMAGE_SIZE, IMAGE_SIZE)
)
cam_to_world = cam_to_world[0].cpu().float().numpy()   # (N, 3, 4)
intrinsics_m = intrinsics_t[0].cpu().float().numpy()   # (N, 3, 3)

# Extend 3×4 → 4×4 by appending [0,0,0,1] bottom row
N_c = cam_to_world.shape[0]
bottom = np.tile(np.array([[[0, 0, 0, 1]]]), (N_c, 1, 1))  # (N, 1, 4)
cam_to_world_44 = np.concatenate([cam_to_world, bottom], axis=1)  # (N, 4, 4)
extrinsics = np.linalg.inv(cam_to_world_44)              # (N, 4, 4) world→cam (COLMAP)

print(f"  {N} camera poses extracted")
print(f"  Intrinsics[0]: fx={intrinsics_m[0,0,0]:.1f}  fy={intrinsics_m[0,1,1]:.1f}")

# ── 5. Extract point cloud ───────────────────────────────────────────────────
pts_flat = predictions["world_points"][0].cpu().float().numpy().reshape(-1, 3)
valid = np.all(np.isfinite(pts_flat), axis=1)
valid &= (np.linalg.norm(pts_flat, axis=1) > 1e-4)
valid &= (np.linalg.norm(pts_flat, axis=1) < 100.0)
pts_flat = pts_flat[valid]
if len(pts_flat) > 200_000:
    idx = np.random.choice(len(pts_flat), 200_000, replace=False)
    pts_flat = pts_flat[idx]
print(f"  Point cloud: {len(pts_flat):,} points")

# ── 6. Write COLMAP workspace ────────────────────────────────────────────────
os.makedirs(f"{COLMAP_DIR}/sparse/0", exist_ok=True)

colmap_imgs = f"{COLMAP_DIR}/images"
if os.path.exists(colmap_imgs): shutil.rmtree(colmap_imgs)
shutil.copytree(FRAMES_DIR, colmap_imgs)

frame_names = sorted([f for f in os.listdir(FRAMES_DIR) if f.endswith(('.png','.jpg'))])
img0 = cv2.imread(os.path.join(FRAMES_DIR, frame_names[0]))
img_h, img_w = img0.shape[:2]

# Scale intrinsics from 224px → original image resolution
fx = float(intrinsics_m[0, 0, 0]) * img_w / IMAGE_SIZE
fy = float(intrinsics_m[0, 1, 1]) * img_h / IMAGE_SIZE
cx = float(intrinsics_m[0, 0, 2]) * img_w / IMAGE_SIZE
cy = float(intrinsics_m[0, 1, 2]) * img_h / IMAGE_SIZE

with open(f"{COLMAP_DIR}/sparse/0/cameras.bin", 'wb') as f:
    f.write(struct.pack('<Q', 1))
    f.write(struct.pack('<i', 1)); f.write(struct.pack('<i', 1))  # id, PINHOLE
    f.write(struct.pack('<Q', img_w)); f.write(struct.pack('<Q', img_h))
    for p in [fx, fy, cx, cy]: f.write(struct.pack('<d', p))

def _rot_to_quat(R):
    t = R[0,0]+R[1,1]+R[2,2]
    if t > 0:
        s=0.5/np.sqrt(t+1); return 0.25/s,(R[2,1]-R[1,2])*s,(R[0,2]-R[2,0])*s,(R[1,0]-R[0,1])*s
    elif R[0,0]>R[1,1] and R[0,0]>R[2,2]:
        s=2*np.sqrt(1+R[0,0]-R[1,1]-R[2,2])
        return (R[2,1]-R[1,2])/s,0.25*s,(R[0,1]+R[1,0])/s,(R[0,2]+R[2,0])/s
    elif R[1,1]>R[2,2]:
        s=2*np.sqrt(1+R[1,1]-R[0,0]-R[2,2])
        return (R[0,2]-R[2,0])/s,(R[0,1]+R[1,0])/s,0.25*s,(R[1,2]+R[2,1])/s
    else:
        s=2*np.sqrt(1+R[2,2]-R[0,0]-R[1,1])
        return (R[1,0]-R[0,1])/s,(R[0,2]+R[2,0])/s,(R[1,2]+R[2,1])/s,0.25*s

n_imgs = min(len(frame_names), N)
with open(f"{COLMAP_DIR}/sparse/0/images.bin", 'wb') as f:
    f.write(struct.pack('<Q', n_imgs))
    for i in range(n_imgs):
        T = extrinsics[i]
        R, t = T[:3,:3], T[:3,3]
        qw,qx,qy,qz = _rot_to_quat(R)
        f.write(struct.pack('<i', i+1))
        for v in [qw,qx,qy,qz,float(t[0]),float(t[1]),float(t[2])]:
            f.write(struct.pack('<d', v))
        f.write(struct.pack('<i', 1))
        f.write(frame_names[i].encode() + b'\x00')
        f.write(struct.pack('<Q', 0))

rgb = np.full((len(pts_flat), 3), 128, dtype=np.uint8)
with open(f"{COLMAP_DIR}/sparse/0/points3D.bin", 'wb') as f:
    f.write(struct.pack('<Q', len(pts_flat)))
    for j, pt in enumerate(pts_flat):
        f.write(struct.pack('<Q', j+1))
        for v in pt:  f.write(struct.pack('<d', float(v)))
        for c in rgb[j]: f.write(struct.pack('<B', int(c)))
        f.write(struct.pack('<d', 0.)); f.write(struct.pack('<Q', 0))

vggt_ply = f"{WORK_DIR}/vggt_pointcloud.ply"
d = np.empty(len(pts_flat), dtype=[('x','f4'),('y','f4'),('z','f4')])
d['x'],d['y'],d['z'] = pts_flat[:,0],pts_flat[:,1],pts_flat[:,2]
PlyData([PlyElement.describe(d,'vertex')]).write(vggt_ply)
SLAM_PLY = vggt_ply

print(f"\n✓ COLMAP workspace ready: {n_imgs} cameras, {len(pts_flat):,} points")

In [ ]:
# Verify VGGT produced a valid COLMAP workspace
cameras_bin = f"{COLMAP_DIR}/sparse/0/cameras.bin"
images_bin  = f"{COLMAP_DIR}/sparse/0/images.bin"
points_bin  = f"{COLMAP_DIR}/sparse/0/points3D.bin"

ok = all(os.path.exists(p) for p in [cameras_bin, images_bin, points_bin])
if not ok:
    raise RuntimeError("COLMAP workspace missing — check the VGGT cell above for errors.")

print(f"✓ cameras.bin  : {os.path.getsize(cameras_bin)} bytes")
print(f"✓ images.bin   : {os.path.getsize(images_bin)} bytes")
print(f"✓ points3D.bin : {os.path.getsize(points_bin)} bytes")
print(f"✓ Ready for gsplat")

---
## 4. Build COLMAP Workspace

In [ ]:
# COLMAP workspace was already written by the VGGT cell above.
# Verify it looks correct before training gsplat.
print(f"COLMAP dir : {COLMAP_DIR}")
print(f"Images dir : {COLMAP_DIR}/images/")
print(f"Sparse dir : {COLMAP_DIR}/sparse/0/")

n_frames = len([f for f in os.listdir(f"{COLMAP_DIR}/images") if f.endswith(('.png','.jpg'))])
print(f"\n  {n_frames} images")
print(f"  cameras.bin  ✓" if os.path.exists(f"{COLMAP_DIR}/sparse/0/cameras.bin") else "  cameras.bin  ✗ MISSING")
print(f"  images.bin   ✓" if os.path.exists(f"{COLMAP_DIR}/sparse/0/images.bin")  else "  images.bin   ✗ MISSING")
print(f"  points3D.bin ✓" if os.path.exists(f"{COLMAP_DIR}/sparse/0/points3D.bin") else "  points3D.bin ✗ MISSING")

---
## 5. Train 3D Gaussian Splatting

In [ ]:
!pip install -q gsplat==1.3.0
!pip install -q tyro viser imageio[ffmpeg] tensorboard \
    "torchmetrics[image]" tqdm scipy nerfview splines pycolmap PyYAML piexif

if not os.path.exists(GSPLAT_REPO):
    !git clone -b v1.3.0 https://github.com/nerfstudio-project/gsplat.git {GSPLAT_REPO}
    !touch {GSPLAT_REPO}/examples/datasets/__init__.py
    !wget -q -O {GSPLAT_REPO}/examples/datasets/colmap.py \
        "https://github.com/nerfstudio-project/gsplat/raw/9b9f98a5b440531376b4a5386aea49f8e820203b/examples/datasets/colmap.py"
    !wget -q -O {GSPLAT_REPO}/examples/exif.py \
        "https://raw.githubusercontent.com/nerfstudio-project/gsplat/main/examples/exif.py"
    !wget -q -O {GSPLAT_REPO}/examples/datasets/exif.py \
        "https://raw.githubusercontent.com/nerfstudio-project/gsplat/main/examples/exif.py"
    !sed -i 's/align_principal_axes/align_principle_axes/g' \
        {GSPLAT_REPO}/examples/datasets/colmap.py
    print("✓ gsplat cloned and patched")
else:
    print("✓ gsplat already exists")

print("Pre-compiling CUDA extensions (~5 min)...")
!rm -rf ~/.cache/torch_extensions/
!MAX_JOBS=2 python -c "import gsplat.cuda._backend; print('✓ CUDA compiled')"

In [ ]:
os.makedirs(GSPLAT_OUTPUT, exist_ok=True)
%cd {GSPLAT_REPO}

!MAX_JOBS=1 python examples/simple_trainer.py default \
    --data_dir {COLMAP_DIR} \
    --data_factor 1 \
    --result_dir {GSPLAT_OUTPUT} \
    --disable_viewer

print("\n✓ gsplat training complete")

In [ ]:
import torch  # explicit import — needed if this cell runs after a kernel restart
from plyfile import PlyData, PlyElement

ckpt_paths = sorted(glob.glob(f"{GSPLAT_OUTPUT}/**/ckpts/*.pt", recursive=True))
if not ckpt_paths:
    print("⚠ No checkpoints found — using SLAM PLY as fallback")
    FINAL_PLY = SLAM_PLY
else:
    ckpt   = torch.load(ckpt_paths[-1], map_location="cpu")
    splats = ckpt["splats"]
    N      = splats["means"].shape[0]

    means     = splats["means"].numpy()
    scales    = splats["scales"].numpy()
    quats     = splats["quats"].numpy()
    opacities = splats["opacities"].numpy()
    sh0       = splats["sh0"].numpy().reshape(N, 3)
    shN_arr   = splats["shN"].numpy().reshape(N, -1) if "shN" in splats else None

    dtype = [('x','f4'),('y','f4'),('z','f4'),('nx','f4'),('ny','f4'),('nz','f4'),
             ('f_dc_0','f4'),('f_dc_1','f4'),('f_dc_2','f4')]
    for i in range(45): dtype.append((f'f_rest_{i}','f4'))
    dtype += [('opacity','f4'),('scale_0','f4'),('scale_1','f4'),('scale_2','f4'),
              ('rot_0','f4'),('rot_1','f4'),('rot_2','f4'),('rot_3','f4')]

    el = np.empty(N, dtype=dtype)
    el['x'],el['y'],el['z'] = means[:,0],means[:,1],means[:,2]
    el['nx']=el['ny']=el['nz']=0
    el['f_dc_0'],el['f_dc_1'],el['f_dc_2'] = sh0[:,0],sh0[:,1],sh0[:,2]
    for i in range(45):
        el[f'f_rest_{i}'] = shN_arr[:,i] if (shN_arr is not None and i < shN_arr.shape[1]) else 0
    el['opacity'] = opacities.flatten()
    el['scale_0'],el['scale_1'],el['scale_2'] = scales[:,0],scales[:,1],scales[:,2]
    el['rot_0'],el['rot_1'],el['rot_2'],el['rot_3'] = quats[:,0],quats[:,1],quats[:,2],quats[:,3]

    FINAL_PLY = f"{GSPLAT_OUTPUT}/splat_final.ply"
    PlyData([PlyElement.describe(el,'vertex')]).write(FINAL_PLY)
    print(f"✓ Exported {N:,} Gaussians → {FINAL_PLY} ({os.path.getsize(FINAL_PLY)/1e6:.1f} MB)")

---
## 6. Grounded-SAM-2 Semantic Masks

In [ ]:
# supervision removed — it caused version conflicts with transformers and was unused
!pip install -q sam2
print("✓ sam2 installed")

In [ ]:
from tqdm import tqdm
from PIL import Image as PILImage
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from sam2.sam2_image_predictor import SAM2ImagePredictor

os.makedirs(MASKS_DIR, exist_ok=True)

TEXT_PROMPT = "chair. table. sofa. monitor. laptop. cup. floor. wall. door. shelf. bed. lamp."

frame_paths = sorted([
    os.path.join(FRAMES_DIR, f)
    for f in os.listdir(FRAMES_DIR)
    if f.endswith(('.jpg','.png'))
])
print(f"Frames: {len(frame_paths)} | Prompt: {TEXT_PROMPT[:50]}...")

gdino_proc  = AutoProcessor.from_pretrained("IDEA-Research/grounding-dino-tiny")
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    "IDEA-Research/grounding-dino-tiny").to("cuda")
print("✓ Grounding DINO loaded")

sam2_predictor = SAM2ImagePredictor.from_pretrained(
    "facebook/sam2.1-hiera-large", device="cuda")
print("✓ SAM 2.1 loaded")

all_masks_info = []
for i, frame_path in enumerate(tqdm(frame_paths)):
    image     = cv2.imread(frame_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w      = image.shape[:2]

    inputs = gdino_proc(
        images=PILImage.fromarray(image_rgb),
        text=TEXT_PROMPT, return_tensors="pt"
    ).to("cuda")
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_proc.post_process_grounded_object_detection(
        outputs, inputs["input_ids"],
        threshold=0.3, text_threshold=0.25,
        target_sizes=[(h, w)]
    )

    boxes  = results[0]["boxes"].cpu().numpy()
    scores = results[0]["scores"].cpu().numpy()
    labels = results[0]["labels"]
    stem   = os.path.basename(frame_path).rsplit('.', 1)[0]

    frame_masks = []
    if len(boxes) > 0:
        sam2_predictor.set_image(image_rgb)
        masks, _, _ = sam2_predictor.predict(box=boxes, multimask_output=False)
        for j in range(len(masks)):
            mfile = f"{stem}_mask_{j:03d}.png"
            cv2.imwrite(os.path.join(MASKS_DIR, mfile),
                        masks[j].squeeze().astype(np.uint8) * 255)
            frame_masks.append({
                "mask_file"  : mfile,
                "label"      : labels[j] if j < len(labels) else "unknown",
                "confidence" : float(scores[j]) if j < len(scores) else 0.0,
                "instance_id": j,
            })

    all_masks_info.append({"frame": os.path.basename(frame_path),
                           "frame_index": i, "masks": frame_masks})

with open(os.path.join(MASKS_DIR, "masks.json"), 'w') as f:
    json.dump(all_masks_info, f, indent=2)

total = sum(len(x["masks"]) for x in all_masks_info)
print(f"\n✓ {total} masks across {len(frame_paths)} frames")

---
## 7. Semantic Lifting — 2D Masks → 3D Gaussians
Projects each Gaussian through every camera, looks up the mask at that pixel,
and assigns the majority-vote label. Previously a Mac-only step; runs fine on Colab.

In [ ]:
from collections import Counter
from plyfile import PlyData, PlyElement

if not FINAL_PLY or not os.path.exists(FINAL_PLY):
    print("⚠ No splat PLY — skipping semantic lifting")
else:
    # ── Load splat ───────────────────────────────────────────
    ply_data  = PlyData.read(FINAL_PLY)
    v         = ply_data['vertex']
    xyz_g     = np.column_stack([v['x'], v['y'], v['z']]).astype(np.float64)
    prop_names = [p for p in v.data.dtype.names if p not in ('x','y','z')]
    props     = {p: np.array(v[p]) for p in prop_names}
    N_g       = len(xyz_g)
    print(f"Splat: {N_g:,} Gaussians")

    # ── Load camera poses from COLMAP ────────────────────────
    def _quat_to_rot(qw,qx,qy,qz):
        return np.array([
            [1-2*(qy**2+qz**2), 2*(qx*qy-qz*qw), 2*(qx*qz+qy*qw)],
            [2*(qx*qy+qz*qw),   1-2*(qx**2+qz**2), 2*(qy*qz-qx*qw)],
            [2*(qx*qz-qy*qw),   2*(qy*qz+qx*qw),   1-2*(qx**2+qy**2)],
        ])

    with open(f"{COLMAP_DIR}/sparse/0/cameras.bin",'rb') as f:
        f.read(8)                          # num_cameras
        f.read(4); f.read(4)               # camera_id, model_id
        img_w_c = struct.unpack('<Q',f.read(8))[0]
        img_h_c = struct.unpack('<Q',f.read(8))[0]
        fx = struct.unpack('<d',f.read(8))[0]
        fy = struct.unpack('<d',f.read(8))[0]
        cx_c = struct.unpack('<d',f.read(8))[0]
        cy_c = struct.unpack('<d',f.read(8))[0]

    K = np.array([[fx,0,cx_c],[0,fy,cy_c],[0,0,1]])

    cameras_list = []
    with open(f"{COLMAP_DIR}/sparse/0/images.bin",'rb') as f:
        num_imgs = struct.unpack('<Q',f.read(8))[0]
        for _ in range(num_imgs):
            f.read(4)  # image_id
            qw,qx,qy,qz = [struct.unpack('<d',f.read(8))[0] for _ in range(4)]
            tx,ty,tz    = [struct.unpack('<d',f.read(8))[0] for _ in range(3)]
            f.read(4)  # camera_id
            name_b = b''
            while True:
                c = f.read(1)
                if c == b'\x00': break
                name_b += c
            n_pts = struct.unpack('<Q',f.read(8))[0]
            f.read(24*n_pts)
            cameras_list.append({'name': name_b.decode(),
                                  'R': _quat_to_rot(qw,qx,qy,qz),
                                  't': np.array([tx,ty,tz])})
    cameras_list.sort(key=lambda x: x['name'])
    print(f"Cameras: {len(cameras_list)}")

    # ── Load masks manifest ──────────────────────────────────
    with open(os.path.join(MASKS_DIR,'masks.json')) as f:
        manifest = json.load(f)

    label_names = {0: 'unlabelled'}
    label_map   = {}
    label_ctr   = 1
    for fi in manifest:
        for mi in fi.get('masks', []):
            lbl = mi['label'].lower().strip()
            if lbl not in label_map:
                label_map[lbl] = label_ctr
                label_names[label_ctr] = lbl
                label_ctr += 1
            mi['_lid'] = label_map[lbl]

    frame_to_masks = {fi['frame']: fi.get('masks',[]) for fi in manifest}

    # ── Project + vote ───────────────────────────────────────
    votes = [Counter() for _ in range(N_g)]
    print(f"Projecting {N_g:,} Gaussians through {len(cameras_list)} cameras...")

    for cam in cameras_list:
        masks_info = frame_to_masks.get(cam['name'], [])
        if not masks_info: continue

        # project all Gaussians
        p_cam = (cam['R'] @ xyz_g.T).T + cam['t']
        vis   = p_cam[:,2] > 0.01
        p_proj = (K @ p_cam.T).T
        z      = np.where(p_cam[:,2:3] > 0.01, p_cam[:,2:3], 0.01)
        pixels = p_proj[:,:2] / z
        vis   &= (pixels[:,0] >= 0) & (pixels[:,0] < img_w_c)
        vis   &= (pixels[:,1] >= 0) & (pixels[:,1] < img_h_c)

        # load mask images
        loaded = []
        for mi in masks_info:
            mp = os.path.join(MASKS_DIR, mi['mask_file'])
            if os.path.exists(mp):
                loaded.append((cv2.imread(mp, cv2.IMREAD_GRAYSCALE), mi['_lid']))

        for idx in np.where(vis)[0]:
            u, vp = int(pixels[idx,0]), int(pixels[idx,1])
            for mimg, lid in loaded:
                if vp < mimg.shape[0] and u < mimg.shape[1] and mimg[vp,u] > 127:
                    votes[idx][lid] += 1
                    break

    final_labels = np.array([
        v.most_common(1)[0][0] if v else 0 for v in votes
    ], dtype=np.int32)

    pct = 100 * np.sum(final_labels > 0) / N_g
    print(f"Labelled: {np.sum(final_labels>0):,}/{N_g:,} ({pct:.1f}%)")
    for lid, cnt in sorted(Counter(final_labels.tolist()).items()):
        print(f"  {label_names.get(lid,'?')}: {cnt:,}")

    # ── Save semantic PLY ────────────────────────────────────
    def _hsv_colour(i, n):
        hsv = np.array([[[int(180*i/max(n,1)), 220, 230]]], dtype=np.uint8)
        return tuple(cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)[0,0].tolist())

    label_colours = {0: (128,128,128)}
    named = [lid for lid in label_names if lid > 0]
    for i, lid in enumerate(sorted(named)):
        label_colours[lid] = _hsv_colour(i, len(named))

    base_dtype = [('x','f4'),('y','f4'),('z','f4')]
    for pn, pa in props.items():
        dt = 'f4' if pa.dtype in (np.float32,np.float64) else \
             'u1' if pa.dtype == np.uint8 else 'f4'
        base_dtype.append((pn, dt))
    base_dtype += [('semantic_label','u2'),
                   ('semantic_r','u1'),('semantic_g','u1'),('semantic_b','u1')]

    out = np.empty(N_g, dtype=base_dtype)
    out['x'],out['y'],out['z'] = xyz_g[:,0].astype('f4'), xyz_g[:,1].astype('f4'), xyz_g[:,2].astype('f4')
    for pn, pa in props.items():
        out[pn] = pa.astype(out[pn].dtype)
    out['semantic_label'] = final_labels.astype(np.uint16)
    for i in range(N_g):
        r,g,b = label_colours.get(int(final_labels[i]), (128,128,128))
        out['semantic_r'][i],out['semantic_g'][i],out['semantic_b'][i] = r,g,b

    PlyData([PlyElement.describe(out,'vertex')], text=False).write(SEMANTIC_PLY)

    lmap_path = SEMANTIC_PLY.replace('.ply','_labels.json')
    with open(lmap_path,'w') as f:
        json.dump({'labels':{str(k):v for k,v in label_names.items()},
                   'colours':{str(k):list(v) for k,v in label_colours.items()}}, f, indent=2)

    print(f"\n✓ Semantic PLY → {SEMANTIC_PLY} ({os.path.getsize(SEMANTIC_PLY)/1e6:.1f} MB)")

---
## 8. CLIP Embeddings — Open-Vocabulary Text Queries
Runs on Colab GPU (faster than Mac MPS). Generates one embedding per semantic
instance so the viewer can answer queries like "chair" or "laptop".

In [ ]:
# openai-clip is the PyPI package — faster and more reliable than git+ install on Colab network
!pip install -q openai-clip
import clip
import torch
from PIL import Image as PILImage

device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    model_clip, preprocess = clip.load("ViT-L/14", device=device)
    print(f"✓ CLIP ViT-L/14 on {device}")
except RuntimeError:
    model_clip, preprocess = clip.load("ViT-B/32", device=device)
    print(f"✓ CLIP ViT-B/32 on {device} (L/14 too large for this GPU)")

with open(os.path.join(MASKS_DIR,'masks.json')) as f:
    manifest_c = json.load(f)

label_crops: dict = {}
for fi in manifest_c:
    fp = os.path.join(FRAMES_DIR, fi['frame'])
    if not os.path.exists(fp):
        base = os.path.splitext(fi['frame'])[0]
        for ext in ['.png','.jpg']:
            alt = os.path.join(FRAMES_DIR, base+ext)
            if os.path.exists(alt): fp = alt; break
    for mi in fi.get('masks',[]):
        lbl = mi['label'].lower().strip()
        mp  = os.path.join(MASKS_DIR, mi['mask_file'])
        if os.path.exists(fp) and os.path.exists(mp):
            label_crops.setdefault(lbl, []).append((fp, mp))

print(f"Labels: {list(label_crops.keys())}")

embeddings = {}
for lbl, pairs in sorted(label_crops.items()):
    if len(pairs) > 20:
        idx = np.linspace(0, len(pairs)-1, 20, dtype=int)
        pairs = [pairs[i] for i in idx]

    feats = []
    for fp, mp in pairs:
        img  = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None: continue
        ys, xs = np.where(mask > 127)
        if len(ys) == 0: continue
        y1,y2,x1,x2 = ys.min(),ys.max(),xs.min(),xs.max()
        if (y2-y1) < 32 or (x2-x1) < 32: continue
        pad = 10
        crop = img[max(0,y1-pad):min(img.shape[0],y2+pad),
                   max(0,x1-pad):min(img.shape[1],x2+pad)]
        inp = preprocess(PILImage.fromarray(crop)).unsqueeze(0).to(device)
        with torch.no_grad():
            f = model_clip.encode_image(inp)
            f = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.cpu().numpy().flatten())

    if feats:
        emb = np.mean(feats, axis=0)
        emb /= np.linalg.norm(emb)
        embeddings[lbl] = emb
        print(f"  {lbl}: {len(feats)} crops → dim={len(emb)}")

save = {}
names, ids = [], []
for i, (lbl, emb) in enumerate(sorted(embeddings.items()), 1):
    save[f"emb_{lbl.replace(' ','_').replace('.','')}"] = emb.astype(np.float32)
    names.append(lbl); ids.append(i)
save['label_names'] = np.array(names, dtype=object)
save['label_ids']   = np.array(ids, dtype=np.int32)
np.savez(EMBEDDINGS, **save)
print(f"\n✓ Embeddings → {EMBEDDINGS}")

---
## 9. Save to Google Drive

In [ ]:
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

def _save(src, dst_name, label):
    if not src or not os.path.exists(src): return
    dst = os.path.join(DRIVE_OUTPUT, dst_name)
    if os.path.isdir(src):
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)
    print(f"✓ {label} → {dst}")

_save(COLMAP_DIR,  "colmap",            "COLMAP workspace")
_save(FINAL_PLY,   "splat_raw.ply",     "Raw splat")
_save(MASKS_DIR,   "masks",             "Semantic masks")
_save(SLAM_PLY,    "slam_pointcloud.ply", "SLAM point cloud")

# Separate existence check to avoid os.path.getsize crash when file is missing
if os.path.exists(SEMANTIC_PLY):
    _save(SEMANTIC_PLY, "splat_semantic.ply",
          f"Semantic splat ({os.path.getsize(SEMANTIC_PLY)/1e6:.0f} MB)")
    sem_json = SEMANTIC_PLY.replace('.ply', '_labels.json')
    if os.path.exists(sem_json):
        _save(sem_json, "splat_semantic_labels.json", "Label mapping")
else:
    print("⚠ No semantic PLY (semantic lifting was skipped)")

if os.path.exists(EMBEDDINGS):
    _save(EMBEDDINGS, "embeddings.npz", "CLIP embeddings")
else:
    print("⚠ No embeddings (CLIP step was skipped)")

if SLAM_TRAJ is not None:
    np.savetxt(os.path.join(DRIVE_OUTPUT,'slam_trajectory.txt'), SLAM_TRAJ, fmt='%.6f')
    print(f"✓ SLAM trajectory → {DRIVE_OUTPUT}/slam_trajectory.txt")

print(f"""
{'='*60}
 DONE — download from Drive:
 {DRIVE_OUTPUT}/

 Place on Mac as:
   splat_semantic.ply          → outputs/{SCENE_NAME}/splat_semantic.ply
   splat_semantic_labels.json  → outputs/{SCENE_NAME}/
   embeddings.npz              → outputs/{SCENE_NAME}/embeddings.npz
   colmap/                     → data/{SCENE_NAME}/colmap/
   masks/                      → data/{SCENE_NAME}/masks/

 Then on Mac:
   python -m viewer.app --scene {SCENE_NAME}
{'='*60}
""")